# M4 — Modèle capteur, RAG cité et agent borné (DiagOps)

## Mission

Ce notebook documente les **choix techniques**, la **stack**, les **données**, le **protocole d'évaluation** et les **décisions** du module 4.

Objectif M4 : décider si l'on peut déployer
1. un **modèle de provenance capteur** (réelle vs fabriquée),
2. un **assistant documentaire** avec citations et abstention,

sous supervision humaine.

> Synthèse alignée sur `docs/`, `src/` et `results/`. Exécutez les cellules pour reproduire les vérifications.

## 1. Stack technique et langages

| Couche | Technologie | Rôle |
|--------|-------------|------|
| Langage | **Python 3.11+** | Pipeline ML + RAG |
| ML capteur | **scikit-learn** (LR, Random Forest) | Classification provenance |
| Features | **pandas**, règles M3 | Agrégats par `window_id` |
| RAG lexical | **rank_bm25** / scoring maison | Retrieval sans embedding |
| RAG vectoriel | **sentence-transformers** | Embeddings `all-MiniLM-L6-v2` |
| Agent | Python pur (`bounded_agent.py`) | 3 actions sans effet externe |
| UI | **Streamlit** (`ui/explorer_app.py`) | Exploration résultats |
| Tests | **pytest** | Contrats, menaces, abstention |
| Config | **YAML** (`configs/`) | Modèle, retrieval, seuils |

### Architecture

```
M3 (règles + features) ──► baseline_m3.py
         │
         ▼
sensor_calibration.csv ──► features.py ──► model_eval.py (LR, RF)
         │
         ▼
knowledge/documents ──► retrieval.py ──► rag_pipeline.py ──► grounded_answer.py
         │
         ▼
bounded_agent.py (answer | search | abstain)
```

## 0. Environnement

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

pd.set_option('display.max_columns', 30)

def find_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'data_pack' / 'MANIFEST.yaml').is_file():
            return root
    raise RuntimeError('Racine dépôt introuvable')

REPO = find_repo_root()
M4_DIR = REPO / 'work' / 'M4'
DATA = Path(os.environ.get('DIAGOPS_DATA_DIR', REPO / 'data_pack' / '2026-S1'))
sys.path.insert(0, str(M4_DIR))

PATHS = {
    'calibration': DATA / 'model_eval' / 'sensor_calibration.csv',
    'test': DATA / 'model_eval' / 'sensor_test.csv',
    'manifest': DATA / 'knowledge' / 'manifest.csv',
    'documents': DATA / 'knowledge' / 'documents',
    'questions': DATA / 'rag_eval' / 'questions.jsonl',
    'm3_ref': DATA / 'reference_runs' / 'm3_for_m4',
}

print('M4   :', M4_DIR)
for name, p in PATHS.items():
    print(f'  {name:12} → {p.exists():5}  {p}')

## 2. Données et règles de nettoyage

### Sources M4

| Fichier | Lignes | Usage | Nettoyage |
|---------|--------|-------|----------|
| `sensor_calibration.csv` | 900 | Entraînement / validation | Features par fenêtre ; **pas de fuite test** |
| `sensor_test.csv` | 1800 | Test scellé (formateur) | **Jamais utilisé pour le réglage** |
| `manifest.csv` + `documents/` | 8 docs | Corpus RAG | Vérification checksums (`io_contracts.py`) |
| `questions.jsonl` | 24 | Éval RAG | Split train/test par champ `split` |
| `m3_for_m4/` | — | Référence pipeline M3 | Lecture seule |

### Clé de groupe obligatoire

**`window_id`** : 30 mesures par fenêtre — le split `GroupShuffleSplit` évite qu'une même fenêtre soit à la fois en train et en validation.

### Pas de « nettoyage » au sens M2/M3

M4 ne corrige pas de CSV métier. Il **construit des features** (`src/features.py`) et **valide les contrats** (`src/io_contracts.py`). Les anomalies capteur sont le **signal** à classifier (provenance fabriquée).

## 3. Aperçu des données capteur

In [ ]:
if PATHS['calibration'].exists():
    cal = pd.read_csv(PATHS['calibration'])
    print('Calibration :', cal.shape)
    print('Colonnes :', list(cal.columns))
    if 'provenance' in cal.columns:
        print('\nRépartition provenance :')
        print(cal['provenance'].value_counts())
    if 'window_id' in cal.columns:
        print('Fenêtres uniques :', cal['window_id'].nunique())
    display(cal.head(3))
else:
    print('sensor_calibration.csv absent')

## 4. Choix des modèles capteur

| Modèle | F1 (calibration) | Verdict |
|--------|------------------|--------|
| Baseline M3 (règles) | ~0,37 | Plancher obligatoire |
| Régression logistique | intermédiaire | Candidat |
| **Random Forest** | **~0,78** | **Retenu** |

**Pourquoi RF ?** Meilleur F1 sur calibration, features mixtes (règles + stats fenêtre), coût faible.

**Pourquoi pas LLM ?** Pas de labels texte ; risque hallucination ; reporté M5+.

Code : `src/baseline_m3.py`, `src/model_eval.py`, `src/features.py`

In [ ]:
results_model = M4_DIR / 'results' / 'benchmark_modele.json'
if results_model.exists():
    bm = json.loads(results_model.read_text(encoding='utf-8'))
    selected = bm.get('selected_model', '?')
    print('Modèle retenu :', selected)
    rows = []
    for name, metrics in bm.get('candidates', {}).items():
        cal = metrics.get('calibration_full', {})
        rows.append({'modèle': name, 'F1': cal.get('f1'), 'recall': cal.get('recall'), 'precision': cal.get('precision')})
    display(pd.DataFrame(rows).sort_values('F1', ascending=False))
else:
    print('Lancez : PYTHONPATH=. python scripts/run_pipeline.py')

## 5. RAG — retrieval et citations

| Stratégie | Principe | Intérêt M4 |
|-----------|----------|------------|
| `none` | Pas de retrieval | Plancher |
| `lexical` | BM25 / scoring mots | Rapide, interprétable |
| `vector` | Embeddings cosine | Meilleur recall@k calibration |

**Réponses citées** : `GroundedAnswer` avec `Citation(document_id, excerpt)`.

**Abstention** : si `answerable=false` ou preuves insuffisantes → pas de réponse inventée.

Code : `src/retrieval.py`, `src/rag_pipeline.py`, `src/grounded_answer.py`

In [ ]:
results_rag = M4_DIR / 'results' / 'benchmark_retrieval.json'
if results_rag.exists():
    rag = json.loads(results_rag.read_text(encoding='utf-8'))
    print('Embedding :', rag.get('embedding_model'))
    for strategy, metrics in rag.get('strategies', {}).items():
        print(f"  {strategy:8} recall@k={metrics.get('recall_at_k')}  abstention={metrics.get('abstention_rate')}")
else:
    print('benchmark_retrieval.json absent — exécutez run_pipeline.py')

## 6. Agent borné et menaces

L'agent (`src/bounded_agent.py`) n'a que **3 actions** :

1. `answer_without_tool` — répondre sans chercher
2. `search_knowledge` — interroger le corpus
3. `abstain` — refuser de répondre

**Aucun effet externe** (pas d'arrêt machine, pas d'écriture BDD).

Tests menaces : `src/threats.py`, `src/brief2/extended_threats.py` (injection, hors périmètre, contradiction…).

In [ ]:
try:
    from src.threats import THREAT_CASES, run_threat_checks
    print(f'{len(THREAT_CASES)} cas de menace définis')
    report = run_threat_checks()
    passed = sum(1 for r in report if r.get('passed'))
    print(f'Tests passés : {passed}/{len(report)}')
except Exception as e:
    print('Import menaces :', e, '— activez le venv M4')

## 7. Validation des contrats knowledge

In [ ]:
if PATHS['manifest'].exists() and PATHS['questions'].exists():
    manifest = pd.read_csv(PATHS['manifest'])
    print('Documents corpus :', len(manifest))
    display(manifest[['document_id', 'title', 'version']].head())

    questions = [json.loads(l) for l in PATHS['questions'].read_text(encoding='utf-8').splitlines() if l.strip()]
    qdf = pd.DataFrame(questions)
    print('\nQuestions RAG :', len(qdf))
    if 'split' in qdf.columns:
        print(qdf['split'].value_counts())
else:
    print('Manifeste ou questions absents')

## 8. Décision et livrables

| Livrable | Fichier |
|----------|--------|
| Dossier conception | `docs/dossier_conception_m4.md` |
| Protocole éval | `docs/protocole_evaluation.md` |
| Threat model | `docs/threat_model.md` |
| Model card | `docs/model_card.md` |
| Matrice décision | `docs/matrice_decision.md` |
| Journal | `journal_bord.md` |

**Décision provisoire (2026-08-31)** : déployer RF + RAG vectoriel **sous conditions** — validation test scellé formateur requise.

## 9. Commandes pour reproduire

```bash
cd work/M4 && source .venv/bin/activate
PYTHONPATH=. python -m pytest -q
PYTHONPATH=. python scripts/run_pipeline.py
PYTHONPATH=. python scripts/run_brief2.py
streamlit run ui/explorer_app.py
python launch.py
```

## 10. Liens amont / aval

- **M3** : règles, features, baseline provenance
- **M5** (futur) : LLM génératif, déploiement online, monitoring